In [42]:
import pandas as pd
import numpy as np
from collections import Counter
from collections import defaultdict

#custom functions
from heuristic_functions import *

##### Assumptions
- 1 year of a tender is allocated 1 year of antigen demand (through 1 or many vaccines); scales appropriately
- Intenvory is based on demand for a given year.
- Ratio of inventory to supply is calculated based on end of year totals (after calcs), but only in the programatic sense (check after math, add inventory at end of year for next year)

##### Load and setup demand

In [43]:
# Load the CSV file
demand_path = 'data/real/antigen_demand_80_20_2_scenarios.csv'
data = pd.read_csv(demand_path)
# Create the two dataframes based on the 'prob' column
demand_80 = data[data['prob'] == 0.8]
demand_20 = data[data['prob'] == 0.2]

demand_80 = demand_80.drop(columns=['prob', 'demand_SID'])
demand_20 = demand_20.drop(columns=['prob', 'demand_SID'])
# Expanding the 'demands' column into 10 separate columns
demand_80_expanded = demand_80['demands'].apply(lambda x: pd.Series(eval(x)))
demand_20_expanded = demand_20['demands'].apply(lambda x: pd.Series(eval(x)))

# Renaming the columns to 1-10
demand_80_expanded.columns = range(1, 11)
demand_20_expanded.columns = range(1, 11)

# Concatenating the expanded demands columns back to the original antigen column
demand_80_final = pd.concat([demand_80['antigen'], demand_80_expanded], axis=1)
#added 11th year to capture any left overdemand at the end of year 10.
demand_80_final[11] = 0.1
demand_20_final = pd.concat([demand_20['antigen'], demand_20_expanded], axis=1)
#added 11th year to capture any left overdemand at the end of year 10.
demand_20_final[11] = 0.1

# demand_80_final.head(), demand_20_final.head()


##### Load and setup Starting Points

In [44]:
file_path = 'data/real/Starting_point.xlsx'

# Load the sheets 'F_start', 'I_start', 'S_start' into their own DataFrames
f_start = pd.read_excel(file_path, sheet_name='F_start')
i_start = pd.read_excel(file_path, sheet_name='I_start')
s_start = pd.read_excel(file_path, sheet_name='S_start')

##### Load pricing data

In [45]:
# Load the Excel file, skipping the first two sheets
money_path = 'data/Vaccine_price_data.xlsx'
sheet_names = pd.ExcelFile(money_path).sheet_names

# Load the remaining sheets into a dictionary of DataFrames
# data = {sheet: pd.read_excel(file_path, sheet_name=sheet) for sheet in sheet_names[2:5]}
price_data = {sheet_names[i]: pd.read_excel(money_path, sheet_name=i).rename(columns=lambda x: "Manufacturer" if x == pd.read_excel(money_path, sheet_name=i).columns[0] else x) for i in range(2, 5)}


##### Load capacity data

In [46]:
# Load the Excel file, only reading the first sheet
capacity_path = 'data/production_capacity_scenarios.xlsx'
sheet_names = pd.ExcelFile(capacity_path).sheet_names

capacity_data = pd.read_excel(capacity_path, sheet_name='base_capacity')
# capacity_data

##### Initialize stuff

In [47]:
#tender length
delta = 3
vaccine_consumption_percent = 1
years = 10

# Creating an empty DataFrame with the specified structure for calculating ratios
antigens = f_start['Antigen']
columns = ['Antigen',1]

ratio_DF = pd.DataFrame(columns=columns)
ratio_DF['Antigen'] = antigens
ratio_DF[1] = np.zeros(len(antigens))

# create DF to store tender schedule
tender_schedules = f_start.copy()

########################################################
#create DF to store current inventory
inventory_DF = i_start.copy()
##########################################################
#create DF to store missed doses
antigens = f_start['Antigen']
columns = ['Antigen','Missed Doses']

missed_doses = pd.DataFrame(columns=columns)
missed_doses['Antigen'] = antigens
missed_doses['Missed Doses'] = np.zeros(len(antigens))

#CREATE TO STORE VACCINES "PURCHASED" THROUGH TENDERS
vaccine_purchases = defaultdict(list)

#track total price throughout
# update for tenders, vaccine data
total_price = 0

#tender cost
tender_cost = 10000


##### Initialize antigens/vaccines/prodicers

In [48]:
A = ["Measles", "Mumps", "Rubella"]
V = ["M", "MR", "MMR"]

A_v = {
    "M": ["Measles"],
    "MR": ["Measles", "Rubella"],
    "MMR": ["Measles", "Mumps", "Rubella"]
}

P = ["Biological_E", 
    "GSK","PT_Bio", 
    "Serum_Institute"
]

P_v = {
    "M": ["Serum_Institute", "PT_Bio"],
    "MR": ["Serum_Institute", "Biological_E"],
    "MMR": ["Serum_Institute", "GSK"]
}

P_v = {
    "M": ["Serum_Institute", "PT_Bio"],
    "MR": ["Serum_Institute", "Biological_E"],
    "MMR": ["Serum_Institute", "GSK"]
}


#translate vaccine - antigen, to antigen - vaccine
V_a = {a: [v for v in A_v if a in A_v[v]] for a in A}

V_p = {p: [v for v in P_v if p in P_v[v]] for p in P}

P_a = {a: list(set(p for v in V_a[a] for p in P_v[v])) for a in A}

A_p = {p: [a for a in P_a if p in P_a[a]] for p in P}



## TESTING - Measles Containing Vaccines Only

In [49]:
# Selecting only the rows for 'Measles', 'Mumps', and 'Rubella' in both datasets
demand_80_MCV = demand_80_final[demand_80_final['antigen'].isin(['Measles', 'Mumps', 'Rubella'])]
interim_demand_DF = demand_80_MCV.copy()
tender_schedules_MCV = tender_schedules[tender_schedules['Antigen'].isin(['Measles', 'Mumps', 'Rubella'])]
# i_start_MCV = i_start[i_start['Vaccine'].isin(['M', 'MR', 'MMR'])]
missed_doses_MCV = missed_doses[missed_doses['Antigen'].isin(['Measles', 'Mumps', 'Rubella'])]
ratio_DF_MCV = pd.DataFrame()
inventory_DF_MCV = inventory_DF[inventory_DF['Vaccine'].isin(['M', 'MR', 'MMR'])]


##### Logic to translate vaccine totals to antigen coverage for later math

In [50]:
#logic to setup least covered antigens:
# Flatten the list of all antigens from all vaccines
all_antigens = [antigen for antigens in A_v.values() for antigen in antigens]
# Count the occurrences of each antigen
antigen_counts = Counter(all_antigens)
# least_covered_antigens = sorted(antigen_counts.keys(), key=lambda x: antigen_counts[x], reverse=False)

In [51]:
inventory_DF_MCV.loc[inventory_DF_MCV['Vaccine'] == 'MMR', 'Amount'].iloc[0]

78464000.0

In [52]:
for year in range(1, 6):  # Iterate through each year - short range for testing  range(1,len(demand_80_MCV.columns)-1)
    print("********************HAPPY NEW YEAR****************************")
    print("******************Reticulating splines...**************************")
    print(f"Year: {year}")

    least_covered_antigens = sorted(antigen_counts.keys(), key=lambda x: antigen_counts[x], reverse=False)
    remaining_inventory = {}
    uncovered_demand = {}
    while least_covered_antigens:  # Iterate through each antigen, find what vaccines cover each antigen, least to greatest, update supply and demand
        print('###############################################################')
        antigen = least_covered_antigens.pop(0)
        print(f"serving antigen {antigen}")
        print(f" this is the check {demand_80_MCV.loc[demand_80_MCV['antigen'] == antigen, year].iloc[0]}")
        for vaccine, antigens in A_v.items():  # Iterate through A_v to check which vaccines cover the antigen
            if antigen in antigens and demand_80_MCV.loc[demand_80_MCV['antigen'] == antigen, year].iloc[0] >0:

                vaccine_inventory_value = inventory_DF_MCV.loc[inventory_DF_MCV['Vaccine'] == vaccine, 'Amount'].iloc[0]
                # print("iiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii")
                # print(f"Inventory for {vaccine} for year {year}: ", vaccine_inventory_value)

                antigen_demand_value = demand_80_MCV.loc[demand_80_MCV['antigen'] == antigen, year].iloc[0]
                # print(f"Demand for {antigen} for year {year}: ", antigen_demand_value)

                difference = vaccine_inventory_value - antigen_demand_value
                if difference >= 0: #
                    remaining_inventory[vaccine] = difference
                    decrement = antigen_demand_value
                else: 
                    remaining_inventory[vaccine] = 0
                    decrement = vaccine_inventory_value
                    uncovered_demand[antigen] = abs(difference)
                    print("------------------------------------------")
                    # print(f"Vaccine:3 {vaccine}, antigen: {antigen}")
                    print(f"uncovered demand for {antigen}: {[antigen]}")
                    #transfer uncovered demand to next year

                # print("^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^")
                inventory_DF_MCV.loc[inventory_DF_MCV.iloc[:, 0] == vaccine, inventory_DF_MCV.columns[1]] = remaining_inventory[vaccine]

                print("Decrementing antigen demands")
                for ant in antigens:
                    if demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == ant, demand_80_MCV.columns[year]].item() > 0:
                        # print(f"from {ant} demand, reducing demand for year {year} for antigen {ant} by {decrement}")
                        demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == ant, demand_80_MCV.columns[year]] = demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == ant].iloc[0, year] - decrement

        print(f"{len(antigen_counts)} antigens entered, only {least_covered_antigens} remain!")

    #update any uncovered demand, to next year. add uncovered demand to dosses_missed dict
    if 'uncovered_demand' in locals(): # Check if the variable exists
        while uncovered_demand:
            top = uncovered_demand.popitem()
            top_antigen = top[0]
            doses_missed = top[1]
            print(f"{doses_missed} doses missed for {top_antigen}")
            demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == top_antigen, demand_80_MCV.columns[year+1]] += doses_missed
            missed_doses_MCV.loc[missed_doses_MCV.iloc[:, 0] == top_antigen, missed_doses_MCV.columns[1]] += doses_missed
            ###################################################################################################################
            #Need to add logic here to update interim_demand_DF[year +1]  with any uncovered demand
            interim_demand_DF.loc[interim_demand_DF.iloc[:,0]==top_antigen]
    else:
        print("no uncovered demand this year")
    
    #check ratio for supply/demand.
    #check at end of year for math reasons. if ratio is less than 1, schedule tender, perform search for vaccines, add inventory
    print(f"Checking ratio of supply to demand for antigens for year {year + 1}!")
    print()
    #pulls the current ratio of supply and demand. returns ratio_DF and antigen coverage DF
    ratio_DF_MCV, coverage_df = calculate_coverage_and_ratios(inventory_DF_MCV, demand_80_MCV, V_a, year+1)

    ratio_DF_MCV = ratio_DF_MCV.sort_values(by='antigen', key=lambda x: x.map(antigen_counts), ascending=True)

    for index, row in ratio_DF_MCV.iterrows():
        if row['Ratio'] < 1.0: #create three year tender
            #update:
            total_price += tender_cost
            # - tender schedule
            # - inventory
            # - missed doses
            # - interim_demand_DF
            print(f"Ratio: {round(row.loc['Ratio'],2)}")
            print(f"Generating Tender for {row.loc['antigen']}")
            #append F schedule for curreny year +1 to current year +1 + tender_length
            new_row = {'Antigen': row.loc['antigen'], 'Starting': year + 1, 'Ending': year + 4}
            tender_schedules_MCV = pd.concat([tender_schedules_MCV, pd.DataFrame([new_row])], ignore_index=True)
            #inventory search

            #this needs to be for a three year period to start
            #then we need to dynamically generate a teneder based on supply/demand over time

            # for year in range(1,tender_length):
            price_list = get_manufacturer_vaccine_price(row.loc['antigen'], price_data, V_a, P_v, year + 1)
            # Display the result
            # print(f"Lowest Price Information: {price_list}")
            #update this dynamically to take in start year and end year is the length of the tender from above that was generated dynamically
            result = fulfill_demand(price_list, row.loc['antigen'], interim_demand_DF, capacity_data, inventory_DF_MCV, year, total_price, A_v) 
            total_price += result['Total_Price']           
            print(result)

        else: #ratio greater than 1
            print(f"Ratio: {round(row.loc['Ratio'],2)}")
            print(f"Supply >= demand for {row.loc['antigen']}")



********************HAPPY NEW YEAR****************************
******************Reticulating splines...**************************
Year: 1
###############################################################
serving antigen Mumps
 this is the check 26052200
Decrementing antigen demands
3 antigens entered, only ['Rubella', 'Measles'] remain!
###############################################################
serving antigen Rubella
 this is the check 331279100
Decrementing antigen demands
3 antigens entered, only ['Measles'] remain!
###############################################################
serving antigen Measles
 this is the check 62765700
Decrementing antigen demands
3 antigens entered, only [] remain!
Checking ratio of supply to demand for antigens for year 2!

Ratio: 1.97
Supply >= demand for Mumps
Ratio: 2.2
Supply >= demand for Rubella
Ratio: 2.1
Supply >= demand for Measles
********************HAPPY NEW YEAR****************************
******************Reticulating splines...******

In [53]:
result['Demand_Filled']

0

In [ ]:
for year in range(1,4):
    print(year)

In [29]:
inventory_DF_MCV

,Vaccine,Amount
4,M,45911731.0
5,MR,0.0
6,MMR,0.0


In [30]:
tender_schedules_MCV

,Antigen,Starting,Ending
0,Measles,1,3
1,Mumps,1,3
2,Rubella,1,3
3,Mumps,4,7
4,Rubella,4,7
5,Measles,4,7
6,Mumps,5,8
7,Rubella,5,8
8,Measles,5,8
9,Mumps,6,9


In [ ]:
vaccine_purchases
